# Ollama on Colab GPU (Devstral Small 2 24B)

Serves a model from a Colab GPU through a public tunnel so your local OrgAI backend can use it.

**Before you start**
1. `Runtime > Change runtime type > GPU`. Free T4 is borderline for this model (see step 5); an L4 or A100 (Colab paid) fits it fully.
2. Run the cells top to bottom. Keep this tab open: when the runtime disconnects, the URL dies.
3. Anyone with the tunnel URL can use your GPU and the model. Don't share it, and stop the runtime when done.
4. Colab's terms restrict some uses of free runtimes, and free sessions can be cut at any time. Treat this as a trial.

## 1. Check the GPU

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(out.stdout.strip() or "NO GPU FOUND. Use Runtime > Change runtime type > GPU, then re-run.")

## 2. Install Ollama

In [ ]:
%%bash
set -e
apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null
curl -fsSL https://ollama.com/install.sh | sh
ollama --version

## 3. Start the Ollama server (local to this Colab VM)

In [ ]:
import os, subprocess, time, urllib.request

MODEL = "devstral-small-2:24b"   # change here to try another model
NUM_CTX = 4096                   # use the SAME value as OLLAMA_NUM_CTX on your machine, or Ollama reloads the model

env = dict(os.environ,
           OLLAMA_HOST="127.0.0.1:11434",
           OLLAMA_KEEP_ALIVE="24h",          # keep the model loaded between requests
           OLLAMA_NUM_PARALLEL="1",
           OLLAMA_MAX_LOADED_MODELS="1",
           OLLAMA_FLASH_ATTENTION="1",       # needed for the quantized KV cache below
           OLLAMA_KV_CACHE_TYPE="q8_0")      # roughly halves KV-cache memory

def start_server():
    global server
    server = subprocess.Popen(["ollama", "serve"], env=env,
                              stdout=open("/content/ollama.log", "a"), stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
            return True
        except Exception:
            time.sleep(1)
    return False

if not start_server():
    raise RuntimeError(open("/content/ollama.log").read()[-2000:])
print("Ollama server is up")

## 4. Pull the model (about 15 GB, takes a few minutes)

In [ ]:
!ollama pull {MODEL}
!ollama list

## 5. Load it and check where it runs

Look at the `PROCESSOR` column of `ollama ps`:
- `100% GPU` is what you want.
- Something like `30%/70% CPU/GPU` means part of the model spilled to the slow CPU (likely on a free T4). It still works but is slower.

In [ ]:
import json

def preload_model():
    req = urllib.request.Request(
        "http://127.0.0.1:11434/api/generate",
        data=json.dumps({"model": MODEL, "prompt": "Say hi.", "stream": False, "keep_alive": "24h",
                         "options": {"num_ctx": NUM_CTX, "num_predict": 8}}).encode(),
        headers={"Content-Type": "application/json"})
    t = time.time()
    urllib.request.urlopen(req, timeout=900).read()
    print(f"model loaded and answered in {time.time() - t:.0f}s")

preload_model()
!ollama ps

## 6. Open the public tunnel (Cloudflare quick tunnel, no account needed)

In [ ]:
import re
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

def start_tunnel():
    # starts a quick tunnel and returns its public URL (a new one every time)
    global tunnel
    # --http-host-header: Ollama (bound to loopback) answers 403 to any Host other than localhost
    tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--no-autoupdate",
                               "--http-host-header", "localhost:11434"],
                              stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
        if m:
            return m.group(0)
        time.sleep(1)
    raise RuntimeError(open("/content/tunnel.log").read()[-2000:])

def show_instructions(url):
    print("PUBLIC URL:", url)
    print()
    print("On your own machine (PowerShell), start the backend like this:")
    print(f'  $env:OLLAMA_HOST = "{url}"')
    print(f'  $env:OLLAMA_NUM_CTX = "{NUM_CTX}"')
    print("  uvicorn app.main:app --reload --port 8765")
    print(f"Then pick 'Local: {MODEL}' in the OrgAI chat.")

PUBLIC_URL = start_tunnel()
show_instructions(PUBLIC_URL)

## 7. Test it through the public URL

Checks that the tunnel works, that the model makes a tool call, and prints the generation speed.

In [ ]:
def call(path, payload=None, timeout=600):
    req = urllib.request.Request(
        PUBLIC_URL + path,
        data=json.dumps(payload).encode() if payload else None,
        headers={"Content-Type": "application/json", "User-Agent": "orgai-test/1.0"})  # Cloudflare blocks the default Python UA
    return json.load(urllib.request.urlopen(req, timeout=timeout))

for _ in range(30):                      # a new tunnel can take a few seconds to resolve
    try:
        print("models:", [m["name"] for m in call("/api/tags")["models"]])
        break
    except Exception as e:
        err = e
        if hasattr(e, "read"):           # HTTPError: show who answered and what they said
            err = f"{e} | {e.headers.get('Server')} | {e.read()[:200]!r}"
        time.sleep(2)
else:
    raise RuntimeError(f"tunnel not reachable: {err}")

r = call("/api/chat", {
    "model": MODEL, "stream": False,
    "messages": [{"role": "user", "content": "Read the file config.py using the tool."}],
    "tools": [{"type": "function", "function": {
        "name": "read_file", "description": "Read a text file in the workspace.",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}}}],
    "options": {"num_ctx": NUM_CTX}})
print("tool_calls:", r["message"].get("tool_calls"))
print(f"generation: {r['eval_count'] / (r['eval_duration'] / 1e9):.1f} tok/s")
print(f"prompt processing: {r['prompt_eval_count'] / max(r['prompt_eval_duration'] / 1e9, 1e-9):.1f} tok/s")

## 8. Keep it running (watchdog)

Leave this cell running while you use the agent. Every 30 s it checks Ollama and the tunnel and restarts whichever died:
- If **Ollama** died, it prints the exit code (`-9` means it was killed from outside, usually out of memory) and the free memory, restarts it and reloads the model. The tunnel URL stays the same.
- If the **tunnel** died, it starts a new one and prints the **new URL**. Copy it to `OLLAMA_HOST` on your machine and restart the backend.

Interrupt it (stop button) when you are done, then run the last cell. Stopping this cell does not stop Ollama or the tunnel.

In [ ]:
def mem_free():
    return subprocess.run(["free", "-m"], capture_output=True, text=True).stdout.strip()

while True:
    now = time.strftime("%H:%M:%S")
    if server.poll() is not None:
        print(f"{now} | Ollama died (exit code {server.poll()}); free memory:\n{mem_free()}")
        if start_server():
            preload_model()
        else:
            print("could not restart Ollama:", open("/content/ollama.log").read()[-1000:])
    if tunnel.poll() is not None:
        PUBLIC_URL = start_tunnel()
        print(f"{now} | tunnel died; started a new one")
        show_instructions(PUBLIC_URL)
    ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout.strip().splitlines()
    print(now, "| alive |", ps[1][:100] if len(ps) > 1 else "model not loaded")
    time.sleep(30)

## 9. Shut down

In [ ]:
tunnel.terminate()
server.terminate()
print("tunnel and server stopped. Also use Runtime > Disconnect and delete runtime to release the GPU.")